In [ ]:
# Install necessary libraries
!pip install sentence-transformers gradio

# Import Libraries
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import gradio as gr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# Upload data files
from google.colab import files

# Reading Plants Growth Data
plants_growth_df = pd.read_csv('/content/plants (1).csv', encoding='latin1')
print("Plants GrowthData has been read.✅")
plants_growth_df.head(2)

Plants GrowthData has been read.✅


,Plant Name,Growth,Soil,Sunlight,Watering,Fertilization Type
0,Aloe Vera,slow,sandy,indirect sunlight,Water weekly,Balanced
1,Basil,fast,well-drained,full sunlight,Keep soil evenly moist,Organic


In [ ]:
# Generate questions, answers, and intents
questions = []
answers = []
intents = []

for index, row in plants_growth_df.iterrows():
    plant = row['Plant Name']

    questions.append(f"What is the best soil for {plant}?")
    answers.append(f"The best soil for {plant} is {row['Soil']}.")
    intents.append("Soil")

    questions.append(f"How often should I water {plant}?")
    answers.append(f"Watering recommendation for {plant}: {row['Watering']}.")
    intents.append("Watering")

    questions.append(f"What sunlight is best for {plant}?")
    answers.append(f"{plant} prefers {row['Sunlight']}.")
    intents.append("Sunlight")

    questions.append(f"What type of fertilizer is recommended for {plant}?")
    answers.append(f"The recommended fertilizer for {plant} is {row['Fertilization Type']}.")
    intents.append("Fertilizer")

    questions.append(f"How fast does {plant} grow?")
    answers.append(f"{plant} has a {row['Growth']} growth rate.")
    intents.append("Growth")

plant_questions_df = pd.DataFrame({
    'question': questions,
    'answer': answers,
    'intent': intents
})

print(" Question and answer data has been created.✅")

 Question and answer data has been created.✅


In [ ]:
# Text cleaning (Preprocessing)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

plant_questions_df['cleaned_question'] = plant_questions_df['question'].apply(clean_text)
print("✅ Texts cleaned")


✅ Texts cleaned


In [ ]:
# Download SBERT form
model = SentenceTransformer('all-MiniLM-L6-v2')

# Question Coding
questions_embeddings = model.encode(plant_questions_df['cleaned_question'].tolist(), show_progress_bar=True)
print(" The questions have been coded.✅")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

 The questions have been coded.✅


In [ ]:
#Building Retrieval Chatbot
def chatbot_response(user_question):
    user_question_clean = clean_text(user_question)
    user_embedding = model.encode([user_question_clean])
    similarities = cosine_similarity(user_embedding, questions_embeddings)[0]

    best_idx = np.argmax(similarities)
    best_question = plant_questions_df.iloc[best_idx]['question']
    best_answer = plant_questions_df.iloc[best_idx]['answer']
    best_intent = plant_questions_df.iloc[best_idx]['intent']
    best_score = similarities[best_idx]

    print(f"\n🔎 Closest Match: {best_question}")
    print(f"🔹 Intent: {best_intent}")
    print(f"🔹 Similarity Score: {round(best_score, 2)}")
    print(f"\n🪴 Answer: {best_answer}\n")
    print("🌱 Feel free to ask more questions about your plants!")


# Trying Chatbot
print("🌟 Chatbot is ready! You can now ask questions in the input below!")

🌟 Chatbot is ready! You can now ask questions in the input below!


In [ ]:
# Intent Classification Model Training
print("✅ Now we start training Intent Classifier...")

✅ Now we start training Intent Classifier...


In [ ]:
# Preparing data for classification
X = plant_questions_df['cleaned_question']
y = plant_questions_df['intent']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert text to numbers using TF-IDF
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Building a classification model
intent_classifier = LogisticRegression(max_iter=1000)
intent_classifier.fit(X_train_vec, y_train)

# prediction
y_pred = intent_classifier.predict(X_test_vec)

# Model Evaluation
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Intent Classifier Accuracy: {round(accuracy*100, 2)}%")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

# Fully prepared
print("\n🚀 Everything is ready! You have a complete Retrieval-based Chatbot and an Intent Classifier!")


✅ Intent Classifier Accuracy: 100.0%

📋 Classification Report:
              precision    recall  f1-score   support

  Fertilizer       1.00      1.00      1.00       122
      Growth       1.00      1.00      1.00       115
        Soil       1.00      1.00      1.00       121
    Sunlight       1.00      1.00      1.00       125
    Watering       1.00      1.00      1.00       113

    accuracy                           1.00       596
   macro avg       1.00      1.00      1.00       596
weighted avg       1.00      1.00      1.00       596


🚀 Everything is ready! You have a complete Retrieval-based Chatbot and an Intent Classifier!


In [ ]:
# Chatbot: Continuous operation
while True:
    user_input = input("\n🌿 Ask me anything about plants (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("👋 Goodbye! Take care of your plants!")
        break
    chatbot_response(user_input)


🌿 Ask me anything about plants (or type 'exit' to quit): How often should I water tomato

🔎 Closest Match: How often should I water Tomato?
🔹 Intent: Watering
🔹 Similarity Score: 1.0

🪴 Answer: Watering recommendation for Tomato: Regular Watering.

🌱 Feel free to ask more questions about your plants!

🌿 Ask me anything about plants (or type 'exit' to quit): exit
👋 Goodbye! Take care of your plants!


In [ ]:
import gradio as gr

# --- define chatbot response function ---
def chatbot_response_gradio(user_question, history):
    user_question_clean = clean_text(user_question)
    user_embedding = model.encode([user_question_clean])
    similarities = cosine_similarity(user_embedding, questions_embeddings)[0]

    best_idx = np.argmax(similarities)
    best_question = plant_questions_df.iloc[best_idx]['question']
    best_answer = plant_questions_df.iloc[best_idx]['answer']
    best_intent = plant_questions_df.iloc[best_idx]['intent']
    best_score = similarities[best_idx]

    response = (
        f"🔎 Closest Match: {best_question}\n"
        f"🔹 Intent: {best_intent}\n"
        f"🔹 Similarity Score: {round(best_score, 2)}\n\n"
        f"🪴 Answer: {best_answer}\n\n"
        f"🌱 Feel free to ask more questions about your plants!"
    )

    history.append((user_question, response))
    return history

# --- build gradio app ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<h1 style='text-align: center; color: #4CAF50;'>🌿 Plant Care Chatbot</h1>")

    chatbot = gr.Chatbot(label="Plant Bot")
    user_input = gr.Textbox(placeholder="Ask about soil, watering, sunlight, etc...", show_label=False)

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear Chat")

    send_btn.click(chatbot_response_gradio, [user_input, chatbot], [chatbot])
    user_input.submit(chatbot_response_gradio, [user_input, chatbot], [chatbot])
    clear_btn.click(lambda: [], None, chatbot)

demo.launch()


<ipython-input-12-18694779cf2e>:30: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Plant Bot")


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37b29cc3381994824e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
